In [1]:
import pandas as pd
import feature_engineering_helper as hf
import importlib
from sklearn.model_selection import StratifiedShuffleSplit



In [2]:
# get raw data
data_prefix = '../0_data/processed_data/'
original_train = pd.read_csv(data_prefix + 'train_without_data_augmentation.csv')[['SMILES','MP']]

original_test = pd.read_csv(data_prefix + 'test_predictions.csv')[['SMILES','exp MP']]
original_test = original_test.rename(columns={'exp MP': 'MP'})

raw_data = pd.concat([original_train, original_test], axis=0).reset_index(drop=True)

print(original_train.shape)
print(original_test.shape)
print(raw_data.shape)

(17633, 2)
(1961, 2)
(19594, 2)


In [3]:
# Clean data with new melting point bounds (between low_bound and high_bound)
clean_data = hf.clean_dataset(raw_data, low_bound=0, high_bound=500)

Initial shape: (19594, 2)
Removed 2370 rows with MP < 0
Removed 2 rows with MP > 500
Final shape: (17222, 2)


In [4]:
# add Ro5 labels
importlib.reload(hf)
clean_data['Ro5'] = clean_data['SMILES'].apply(hf.check_Ro5)
# how many molecules in percentage violate Ro5
violation_percentage = 100 * (clean_data['Ro5'] == False).mean()
print(f"{violation_percentage:.2f}% of molecules violate Ro5")
clean_data.describe()

2.13% of molecules violate Ro5


,MP,Ro5
count,17222.000000,17222.00000
mean,126.045995,0.97869
std,70.913439,0.14442
min,0.000000,0.00000
25%,69.000000,1.00000
50%,120.000000,1.00000
75%,175.000000,1.00000
max,492.500000,1.00000


In [5]:
data_with_features = hf.smiles_to_features(clean_data, ['rdkit', 'maccs'])
data_with_features

✓ RDKit: Added 208 features
✓ MACCS: Added 167 features


,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,Ro5,SMILES
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.461644,1,CCOC(=O)/C=C/C(=O)OCC
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0.322574,1,O=C(c1ccc(cc1)C(=O)Oc1cc(Cl)cc(c1)Cl)Oc1cc(Cl)...
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.344216,1,O=C(c1ccccc1)Nc1cccc(c1)/N=N/c1cccc(c1)NC(=O)c...
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.609080,1,OCc1cccc(n1)CO
5,0,0,0,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0.779142,1,COc1ccc(cc1)c1nnc2n1NC(S2)c1ccc(cc1)Cl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19588,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0.735003,1,OC(=O)Cc1ccc(c(c1)F)F
19589,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0.812970,1,CCOC(=O)N(c1ccccc1)c1ccccc1
19591,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0.805020,1,OC(=O)Cc1ccc(c(c1)Cl)Cl
19592,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.519957,1,CCCOC(=O)c1cc(O)c(c(c1)O)O


In [6]:
# drop rows with NaN values and print how many rows were dropped and reset index
nan_rows = data_with_features[data_with_features.isna().any(axis=1)]
num_dropped = nan_rows.shape[0]
print(f"Dropped {num_dropped} rows containing NaN values.")
data_with_features = data_with_features.dropna().reset_index(drop=True)

Dropped 2 rows containing NaN values.


In [7]:
# stratified split based on Ro5 compliance

split = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_index, test_index in split.split(data_with_features, data_with_features['Ro5']):
    strat_train_set = data_with_features.loc[train_index]
    strat_test_set = data_with_features.loc[test_index]
    strat_train_set['Type'] = 'Train'
    strat_test_set['Type'] = 'Test'
final_data = pd.concat([strat_train_set, strat_test_set], axis=0).reset_index(drop=True)

# print the shape of train and test dataset
print(f"Train set: {final_data[final_data['Type'] == 'Train'].shape}")
print(f"Train set: {final_data[final_data['Type'] == 'Test'].shape}")

# print the percentage of Ro5 violations in train and test sets
train_violation_percentage = 100 * (final_data[final_data['Type'] == 'Train']['Ro5'] == False).mean()
test_violation_percentage = 100 * (final_data[final_data['Type'] == 'Test']['Ro5'] == False).mean()
print(f"Train set: {train_violation_percentage:.2f}% of molecules violate Ro5")
print(f"Test set: {test_violation_percentage:.2f}% of molecules violate Ro5")


Train set: (12054, 379)
Train set: (5166, 379)
Train set: 2.13% of molecules violate Ro5
Test set: 2.13% of molecules violate Ro5


In [8]:
final_data.to_parquet(data_prefix + 'data_with_all_features.parquet', compression= "zstd")

In [9]:
import pickle
from sklearn.preprocessing import StandardScaler

def standardize_data(non_feature_cols, dataframe, scaler_path):

    df_scaled = dataframe.copy()
    
    # Identify feature columns (exclude non-feature columns)

    feature_cols = [col for col in df_scaled.columns if col not in non_feature_cols]
    
    print(f"Number of feature columns to standardize: {len(feature_cols)}")
 
    # Generate scaler using only training data
    train_df = df_scaled[df_scaled['Type'] == 'Train']
    
    if len(train_df) == 0:
        raise ValueError(f"No training data found. 'Type' column values: {df_scaled['Type'].unique()}")
        
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])
    
    if scaler_path is not None:
        # Save the scaler
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler, f)
        print(f"Scaler saved to {scaler_path}")
    
    # Apply to the whole dataset
    df_scaled[feature_cols] = scaler.transform(df_scaled[feature_cols])

    
    return df_scaled

In [10]:
data_with_all_features_scaled = hf.standardize_data(non_feature_cols=['SMILES', 'MP', 'Ro5', 'Type'], dataframe=final_data, scaler_path=data_prefix + 'data_with_all_features_scaled_scaler.pkl')
data_with_all_features_scaled.to_parquet(data_prefix + 'data_with_all_features_scaled.parquet', compression= "zstd")



Number of feature columns to standardize: 375
Scaler saved to ../0_data/processed_data/data_with_all_features_scaled_scaler.pkl


In [11]:
data_with_all_features_scaled[data_with_all_features_scaled['Type'] == 'Train'].describe()

,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_sulfone,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,Ro5
count,12054.0,12054.0,12054.0,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,...,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,12054.000000
mean,0.0,0.0,0.0,-2.269445e-17,3.242065e-18,1.002093e-17,2.947332e-17,-4.804151e-17,5.010464e-17,-4.715731e-17,...,2.770492e-17,-4.126264e-17,-2.549442e-17,-3.242065e-17,1.473666e-17,-2.475759e-17,-4.479944e-17,-5.187304e-17,7.957796e-17,0.978679
std,0.0,0.0,0.0,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,...,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,0.144457
min,0.0,0.0,0.0,-3.731110e-01,-6.185924e-01,-4.833742e-01,-4.413237e-01,-4.301368e-01,-6.390165e-01,-4.804989e-01,...,-1.037811e-01,-6.891473e-02,-5.548848e-02,-1.279550e-01,-2.430697e-02,-1.246099e-01,-1.737564e-01,-1.489146e-01,-3.353023e+00,0.000000
25%,0.0,0.0,0.0,-3.731110e-01,-6.185924e-01,-4.833742e-01,-4.413237e-01,-4.301368e-01,-6.390165e-01,-4.804989e-01,...,-1.037811e-01,-6.891473e-02,-5.548848e-02,-1.279550e-01,-2.430697e-02,-1.246099e-01,-1.737564e-01,-1.489146e-01,-6.132058e-01,1.000000
50%,0.0,0.0,0.0,-3.731110e-01,-6.185924e-01,-4.833742e-01,-4.413237e-01,-4.301368e-01,-6.390165e-01,-4.804989e-01,...,-1.037811e-01,-6.891473e-02,-5.548848e-02,-1.279550e-01,-2.430697e-02,-1.246099e-01,-1.737564e-01,-1.489146e-01,8.107313e-02,1.000000
75%,0.0,0.0,0.0,-3.731110e-01,1.616573e+00,-4.833742e-01,-4.413237e-01,-4.301368e-01,1.564905e+00,-4.804989e-01,...,-1.037811e-01,-6.891473e-02,-5.548848e-02,-1.279550e-01,-2.430697e-02,-1.246099e-01,-1.737564e-01,-1.489146e-01,7.196050e-01,1.000000
max,0.0,0.0,0.0,2.680168e+00,1.616573e+00,2.068791e+00,2.265911e+00,2.324842e+00,1.564905e+00,2.081170e+00,...,1.829295e+01,4.739955e+01,1.802176e+01,1.529574e+01,5.324774e+01,2.143588e+01,1.762619e+01,1.300139e+01,2.179419e+00,1.000000
